# Urban Intelligence Platform: Rainfall Prediction & Station K-Means Clustering
**Chennai Real Historical Daily Precipitation Telemetry (1993 - 2023)**

This notebook trains:
1. A **RandomForestRegressor** pipeline on historical daily observations from 62 Chennai meteorological stations.
2. A **K-Means Clustering Pipeline** (`KMeans(n_clusters=3)`) grouping Chennai stations into empirical precipitation zones (Lower, Moderate, Heavy Surge).
3. Exports `rainfall_prediction_pipeline.joblib` and `rainfall_kmeans_pipeline.joblib`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Load and Clean Chennai Rainfall Dataset

In [ ]:
df = pd.read_csv('chennai_rainfall.csv')
print('Shape:', df.shape)
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['Date', 'Rainfall', 'Station']).drop_duplicates(subset=['Station', 'Date'])
df = df.sort_values(by=['Station', 'Date']).reset_index(drop=True)
df['Rainfall'] = pd.to_numeric(df['Rainfall'], errors='coerce').fillna(0)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter
df['Prev_Day_Rainfall'] = df.groupby('Station')['Rainfall'].shift(1).fillna(0)
df['Prev_7_Day_Avg'] = df.groupby('Station')['Rainfall'].rolling(7, min_periods=1).mean().reset_index(0, drop=True).fillna(0)
df.head()


## 2. Train Rainfall Random Forest Regressor Pipeline

In [ ]:
feature_cols = ['Station', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Prev_Day_Rainfall', 'Prev_7_Day_Avg']
X = df[feature_cols]
y = df['Rainfall'].values

split_idx = int(len(df) * 0.85)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Station']),
    ('num', 'passthrough', ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'Prev_Day_Rainfall', 'Prev_7_Day_Avg'])
])

rainfall_rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1))
])

rainfall_rf_pipeline.fit(X_train, y_train)
pred = np.maximum(rainfall_rf_pipeline.predict(X_test), 0)

print('MAE:', round(mean_absolute_error(y_test, pred), 2), 'mm')
print('RMSE:', round(mean_squared_error(y_test, pred) ** 0.5, 2), 'mm')
print('R2 Score:', round(r2_score(y_test, pred), 4))


## 3. Station K-Means Spatial Precipitation Zones

In [ ]:
station_profile = df.groupby('Station').agg(
    mean_rainfall=('Rainfall', 'mean'),
    max_rainfall=('Rainfall', 'max'),
    heavy_rain_days=('Rainfall', lambda x: (x >= 64.5).sum()),
    moderate_rain_days=('Rainfall', lambda x: ((x >= 15.6) & (x < 64.5)).sum())
).reset_index()

station_clust_features = ['mean_rainfall', 'max_rainfall', 'heavy_rain_days', 'moderate_rain_days']
rainfall_kmeans_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=3, random_state=42, n_init=10))
])

station_profile['Cluster'] = rainfall_kmeans_pipeline.fit_predict(station_profile[station_clust_features])
print(station_profile.groupby('Cluster')[station_clust_features].mean())


## 4. Export Pipelines for FastAPI Serving

In [ ]:
joblib.dump({'pipeline': rainfall_rf_pipeline}, 'rainfall_prediction_pipeline.joblib')
joblib.dump({'pipeline': rainfall_kmeans_pipeline}, 'rainfall_kmeans_pipeline.joblib')
print('Rainfall pipelines exported successfully!')
